<a href="https://colab.research.google.com/github/JuanZapa7a/Medical-Image-Processing/blob/main/PIM_Challenge/PIM_Challenge_Student_Practica_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UPCT Medical Image Segmentation Challenge 2026-27
## Práctica 8

**Asignatura:** Procesado de Imágenes Médicas (521104007)

**Profesor:** Juan Zapata

> Contenido **nuevo** de esta práctica. Copia las celdas de aquí abajo y pégalas **al final** de tu propio notebook (el que empezaste en la Práctica 6) — no repitas las prácticas anteriores, ya las tienes hechas ahí.

## Guía de Sesiones (2 horas por sesión)
| Práctica | Fechas (Grupo A / B) | Objetivo de la Sesión | Checkpoint Visual |
|----------|----------------------|-----------------------|-------------------|
| **P6** | 28 Oct - 2 Nov | EDA, Dataset y formato RLE | 6 imágenes con máscaras + RLE OK |
| **P7** | 9-11 Nov | Baseline U-Net y 1ª Submission | Gráficas de Loss + Submission Kaggle |
| ▶ **P8** | 16-18 Nov | Data Augmentation y mejora | Comparativa Baseline vs Augmented |
| **P9** | 23-25 Nov | Inferencia, Threshold y Errores | 5 imágenes normales + 2 casos de error |
| **P10** | 30 Nov-2 Dic | TTA, Submission Final y Defensa | Mejor Dice Score + Defensa Oral |

> **Regla de Oro:** Según el Art. 7.5 del Reglamento de Evaluación UPCT, la asistencia y validación del Checkpoint en el aula es obligatoria para superar la práctica.


# Práctica 8: Data Augmentation y Mejora del Modelo
## Sesión única (16 Nov Grupo A / 18 Nov Grupo B)

### Objetivos de la sesión:
1. Entender las técnicas de Data Augmentation específicas para imágenes médicas
2. Implementar un pipeline de augmentation con Albumentations
3. Re-entrenar el modelo con datos augmentados
4. Comparar resultados: Baseline vs Augmented

### Reglas de oro en imágenes médicas:
- **SÍ hacer**: Rotaciones pequeñas (±15° o menos), flips, elastic deformations suaves, ajuste de brillo/contraste
- **NO hacer**: Rotaciones de 90° (pierde orientación anatómica), crops agresivos (pérdida de contexto), deformaciones extremas

> **CHECKPOINT P8:** Mostrar al profesor:
> 1. Visualización de 4-6 transformaciones aplicadas a una misma imagen
> 2. Tabla comparativa: Baseline vs Augmented (Dice Score)

## Bloque 8.1: Data Augmentation en Imágenes Médicas
### De la memorización a la generalización

En el Bloque 7.3 vimos esta señal de alarma:

| Situación | Loss | Dice | Interpretación |
|-----------|------|------|-----------------|
| Divergencia train/val | Train baja, Val sube | Train sube, Val baja | Señal de sobreajuste |

Con solo 546 imágenes de entrenamiento, un modelo con millones de parámetros (ResNet34 + decoder) tiene capacidad de sobra para **memorizar** las imágenes concretas que ve, en vez de aprender patrones que **generalicen** a imágenes nuevas. El síntoma es exactamente esa divergencia: el modelo mejora en train pero deja de mejorar (o empeora) en val.

### Data Augmentation como regularización

La idea no es conseguir más pacientes ni más ecografías reales: es fabricar **variaciones artificiales** de las imágenes que ya tenéis, para que el modelo nunca vea exactamente la misma imagen dos veces.

| Sin augmentation | Con augmentation |
|---|---|
| El modelo puede memorizar los 437 pares imagen-máscara exactos | Cada época ve versiones ligeramente distintas (rotadas, más claras, con ruido...) |
| Aprende también el ruido específico de esas imágenes | Solo sobrevive lo que es realmente **invariante** (la forma del tumor, no el brillo exacto de esa ecografía) |

> **Idea clave:** Data Augmentation no añade información nueva sobre el mundo — obliga al modelo a no depender de detalles irrelevantes de cada imagen concreta.

### Dos familias de transformaciones

En el Bloque 7.1 ya distinguimos dos tipos de transformación en un `Dataset` de segmentación, y aquí es donde importa de verdad:

| Tipo | Ejemplos | ¿Se aplica a la máscara? |
|------|----------|----------------------------|
| Geométrica | Flip, rotación, shift/scale, elastic transform | Sí, exactamente igual que a la imagen |
| Fotométrica | Brillo, contraste, ruido gaussiano | No — la máscara sigue siendo binaria (0/1) |

Si rotáis la imagen y no la máscara (o al revés), el modelo entrena con pares desalineados: aprenderá a segmentar mal, con un Dice que se desploma sin ningún error de código que lo delate.

### Por qué no todas las transformaciones valen en imagen médica

Una rotación de 90° o un flip vertical son transformaciones perfectamente válidas para fotos de gatos, pero no aquí:

| Transformación | Problema clínico |
|-----------------|-------------------|
| Rotación de 90°/180° | En ecografía mamaria la orientación de la sonda es clínicamente relevante; rotar así crea imágenes que no corresponden a ninguna adquisición real |
| Crops agresivos | Pueden eliminar el tumor por completo de la imagen sin eliminarlo de la máscara |
| Deformaciones elásticas extremas | Un tumor deformado más allá de lo fisiológicamente posible enseña al modelo formas que no existen en la realidad |

La regla general: **una transformación solo es válida si el resultado podría haber sido una adquisición real** en la consulta. Por eso las rotaciones se limitan a ±10-15° (variación plausible en el ángulo de la sonda) y no a giros completos.

### Cómo se compone un pipeline en Albumentations

```python
train_augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])
```

Dos detalles que suelen pasar desapercibidos:

- Cada `p=` es una probabilidad **independiente** por transformación: `p=0.5` no significa "la mitad de las imágenes se transforman", significa que *esa* transformación concreta se aplica con un 50% de probabilidad, combinándose con las demás.
- `A.Normalize` + `ToTensorV2` hacen automáticamente lo que en el Bloque 8.1 escribisteis a mano (normalización ImageNet + conversión a tensor `(C,H,W)`). Por eso a partir de aquí ya no repetiréis ese código manualmente.

### Aumentar solo el train, nunca val ni test

`train_augment` se aplica exclusivamente al `DataLoader` de entrenamiento. El de validación sigue usando el preprocesamiento simple del Bloque 8.1.

> **Pregunta para pensar:** si aplicarais también augmentation al conjunto de validación, ¿seguiría siendo una medida fiable de cómo generaliza el modelo a datos nuevos, o os estaríais engañando a vosotros mismos?

La razón es simple: val y test deben reflejar la distribución real que el modelo verá en producción (una ecografía tal cual la toma el equipo, no una versión rotada y con ruido añadido). Si medís sobre datos artificialmente aumentados, la métrica deja de ser comparable con el Leaderboard de Kaggle.

> **Pregunta para pensar:** entrenar con augmentation añade variabilidad extra en cada época — ¿esperaríais que el modelo converja igual de rápido que el baseline de la Práctica 7, con el mismo número de épocas, o necesitaría más tiempo para notarse la mejora?

### Resumen rápido

| Concepto | Idea principal |
|----------|-----------------|
| Overfitting | Train mejora, val se estanca o empeora — señal de memorización |
| Data Augmentation | Regularización: variaciones artificiales que evitan memorizar detalles irrelevantes |
| Transf. geométrica | Se aplica igual a imagen y máscara |
| Transf. fotométrica | Se aplica solo a la imagen |
| Reglas médicas | Solo transformaciones que podrían corresponder a una adquisición real |
| `p=` en Albumentations | Probabilidad independiente por transformación, no fracción del dataset |
| Alcance | Augmentation solo en train; val/test se quedan con el preprocesamiento simple |

### Referencias

1. **Shorten, C., & Khoshgoftaar, T. M. (2019).** *A survey on Image Data Augmentation for Deep Learning.* Journal of Big Data.
2. **Buslaev, A., et al. (2020).** *Albumentations: Fast and Flexible Image Augmentations.* Information.

## Tarea 8.1: ¿Por qué Data Augmentation?
### El problema del sobreajuste (Overfitting)
Cuando entrenamos con pocos datos (546 imágenes), el modelo tiende a **memorizar** en lugar de **generalizar**. Esto se manifiesta como:
- Train Loss baja mucho
- Val Loss se estanca o sube
- Train Dice alto, Val Dice bajo

### La solución: Data Augmentation
Consiste en crear **variaciones artificiales** de las imágenes de entrenamiento para:
1. Aumentar el tamaño efectivo del dataset
2. Forzar al modelo a aprender características invariantes
3. Reducir el sobreajuste

### Transformaciones específicas para ultrasonido mamario:

| Transformación | Parámetro recomendado | Justificación clínica |
|----------------|----------------------|----------------------|
| **Horizontal Flip** | p=0.5 | El seno izquierdo/derecho es simétrico |
| **Vertical Flip** | p=0.3 | Menos común pero válido |
| **Rotación** | ±15° | La sonda puede tener ángulos variados |
| **Elastic Transform** | alpha=1, sigma=50 | Simula deformaciones del tejido |
| **Gaussian Noise** | var_limit=(10,50) | Simula ruido del equipo |
| **Brightness/Contrast** | ±0.2 | Variaciones en la ganancia del equipo |

>  **IMPORTANTE**: Las máscaras deben transformarse **igual** que las imágenes para mantener la alineación.

In [ ]:
# ============================================================
# TAREA 8.1: PIPELINE DE DATA AUGMENTATION
# ============================================================
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ESCRIBE TU CÓDIGO AQUÍ
# 1. Define un pipeline de augmentation con A.Compose()
# 2. Incluye al menos estas transformaciones:
#    - Resize a IMG_SIZE
#    - HorizontalFlip (p=0.5)
#    - VerticalFlip (p=0.3)
#    - RandomRotate90 (p=0.3)
#    - ShiftScaleRotate (shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5)
#    - ElasticTransform (alpha=1, sigma=50, p=0.3)
#    - GaussNoise (var_limit=(10, 50), p=0.3)
#    - RandomBrightnessContrast (brightness_limit=0.2, contrast_limit=0.2, p=0.5)
#    - Normalize (mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
#    - ToTensorV2()

train_augment = A.Compose([
    # Tu código aquí...
])

print("Pipeline de augmentation definido")

## Bloque 8.2: Verificar Visualmente un Pipeline de Augmentation
### Por qué visualizar antes de entrenar

Un pipeline de Albumentations mal configurado no lanza ningún error — simplemente entrena con datos deformados de forma silenciosa. Descubrir esto después de 30 épocas de entrenamiento (varios minutos u horas de GPU) es mucho más caro que descubrirlo mirando una figura con 6 ejemplos antes de empezar. Por eso, en cualquier pipeline de augmentation nuevo, la primera comprobación siempre es visual, no numérica.

### Cada llamada al pipeline es aleatoria

`train_augment` no es una función determinista: cada vez que la llamáis sobre la misma imagen, el resultado es distinto. Esto ocurre porque, en cada llamada, el pipeline:

1. Decide para cada transformación si se aplica o no, según su `p=` (una tirada de moneda independiente por transformación).
2. Si se aplica, muestrea sus parámetros al azar dentro del rango configurado (por ejemplo, `A.Rotate(limit=10)` elige un ángulo aleatorio entre -10° y +10°, no siempre el mismo).

| Llamada | HorizontalFlip (p=0.5) | Rotate (limit=10) |
|---------|--------------------------|---------------------|
| 1 | No se aplica | +7° |
| 2 | Se aplica | -3° |
| 3 | Se aplica | +10° |

Por eso, al llamar a `train_augment` 6 veces sobre la misma imagen de entrada, obtenéis 6 salidas distintas — es el comportamiento esperado, no un bug. Durante el entrenamiento real esto es justo lo que queréis: cada época, cada imagen se transforma de forma distinta.

### El pipeline termina en tensor normalizado: hay que deshacerlo para poder verlo

Si recordáis el Bloque 8.1, `train_augment` termina con `A.Normalize(...)` y `ToTensorV2()`. Eso significa que lo que devuelve **no es una imagen visualizable directamente**:

| Propiedad | Imagen "normal" (para `plt.imshow`) | Salida de `train_augment` |
|-----------|----------------------------------------|------------------------------|
| Forma | `(H, W, C)` | `(C, H, W)` |
| Rango de valores | `[0, 1]` o `[0, 255]` | Centrado en 0, puede ser negativo |
| Tipo | `numpy.ndarray` | `torch.Tensor` |

Para poder visualizarla hay que deshacer exactamente esos tres pasos, en orden inverso:

```python
aug_img = augmented['image'].permute(1, 2, 0).numpy()          # (C,H,W) -> (H,W,C)
aug_img = aug_img * std + mean                                  # deshacer la normalización ImageNet
aug_img = np.clip(aug_img, 0, 1)                                 # errores de redondeo pueden salirse de [0,1]
```

> **Nota:** el `np.clip` no es cosmético — sin él, `matplotlib` puede mostrar colores extraños o lanzar un aviso, porque tras la operación `* std + mean` algunos píxeles pueden quedar ligeramente fuera de `[0, 1]` por precisión numérica.

### Qué mirar en las imágenes generadas

No basta con comprobar que "se ve algo distinto" en cada versión. Revisad específicamente:

- ¿Sigue siendo reconocible la anatomía, o la transformación es tan agresiva que ya no parece una ecografía real?
- ¿Han aparecido bordes negros o artefactos por el relleno (`border_mode`) que usa la rotación o el shift al salirse del marco original?
- ¿Alguna combinación de transformaciones (rotación + brillo + ruido a la vez) produce una imagen irreconocible, aunque cada transformación individual pareciera razonable?

> **Pregunta para pensar:** si al ver las 6 imágenes augmentadas 2 de ellas os parecen clínicamente irreconocibles (no se distingue tejido de fondo), ¿qué parte del pipeline del Bloque 8.1 revisaríais primero?

### Resumen rápido

| Concepto | Idea principal |
|----------|-----------------|
| Visualizar antes de entrenar | Detecta pipelines rotos sin gastar horas de entrenamiento |
| Aleatoriedad | Cada llamada resuelve de nuevo el `p=` y remuestrea parámetros; 6 llamadas → 6 resultados distintos |
| Desnormalizar | `permute` para volver a `(H,W,C)`, deshacer `Normalize` con `*std + mean`, y `clip` a `[0,1]` |
| Qué revisar | Anatomía reconocible, artefactos de borde, combinaciones demasiado agresivas |

## Tarea 8.2: Visualización de las Transformaciones
Antes de re-entrenar, vamos a visualizar cómo se ven las imágenes con las transformaciones aplicadas.

### Instrucciones:
1. Carga una imagen de ejemplo del dataset de train
2. Aplica el pipeline `train_augment` 6 veces a la misma imagen
3. Muestra la imagen original + 6 versiones augmentadas en una figura de 1x7
4. Observa cómo cambian: rotación, brillo, deformación, etc.

> **CHECKPOINT P8.1:** Muestra al profesor la figura con las 7 imágenes (original + 6 augmentadas)

In [ ]:
# ============================================================
# TAREA 8.2: VISUALIZACIÓN DE AUGMENTATIONS
# ============================================================
import cv2
import matplotlib.pyplot as plt
import numpy as np

# ESCRIBE TU CÓDIGO AQUÍ
# 1. Carga una imagen de ejemplo (usa df_train.iloc[0]['image_path'])
# 2. Crea una figura de 1 fila x 7 columnas
# 3. Muestra la imagen original en la primera columna
# 4. Aplica train_augment 6 veces y muestra cada resultado

# Tu código aquí...

plt.tight_layout()
plt.show()

## Bloque 8.3: Repetir un Experimento de Forma Justa
### Por qué reinicializar el modelo en vez de continuar entrenando

`model_aug = smp.Unet(...)` crea un modelo **nuevo desde cero**, no continúa entrenando el `model` de la Práctica 7. Esto no es un descuido: si continuarais entrenando el modelo baseline ya convergido, cualquier mejora en el Dice no podríais atribuirla al augmentation — podría deberse simplemente a que el modelo llevaba más épocas entrenando en total.

> **Idea clave:** para que la comparación Baseline vs Augmented signifique algo, ambos modelos deben partir de la misma situación de salida (arquitectura recién inicializada) y diferir en **una sola cosa**: los datos que ven durante el entrenamiento.

### Variable de control: cambiar una cosa, no dos

Esto es el mismo principio que un experimento controlado en cualquier ciencia experimental:

| Elemento | Baseline (P7) | Augmented (P8) | ¿Cambia? |
|----------|----------------|-------------------|-----------|
| Arquitectura (`smp.Unet(...)`) | ResNet34 + decoder | ResNet34 + decoder | No |
| `NUM_EPOCHS` | El mismo valor | El mismo valor | No |
| Optimizador y `lr` | `Adam`, `lr=1e-4` | `Adam`, `lr=1e-4` | No |
| Función de pérdida | `DiceBCELoss` | `DiceBCELoss` | No |
| Datos de entrenamiento | Sin augmentation | Con `train_augment` | **Sí — la única variable** |

Si cambiarais también el número de épocas o el learning rate al mismo tiempo que añadís augmentation, y el Dice mejora, no sabríais si mejoró por el augmentation o por el otro cambio. Un experimento con dos variables a la vez no permite sacar conclusiones de ninguna de las dos.

### El riesgo real de "reutilizar el código de P7"

La instrucción de la tarea dice literalmente "copia el código de entrenamiento de P7 y modifica solo el DataLoader". Ese copia-pega es exactamente donde se cuela el bug más común de esta práctica: si olvidáis renombrar alguna variable, sobrescribís sin querer los resultados del baseline.

| Variable original (P7) | Variable nueva (P8) | Si os olvidáis de renombrar... |
|---|---|---|
| `model` | `model_aug` | Seguís entrenando/evaluando el modelo baseline, no uno nuevo |
| `optimizer` | `optimizer_aug` | El optimizador sigue ligado a los parámetros del modelo baseline |
| `train_loader` | `train_loader_aug` | Entrenáis otra vez sin augmentation, aunque creáis que sí lo tiene |
| `train_losses`, `val_losses`, `train_dices`, `val_dices` | `..._aug` | **Sobrescribís las listas del baseline** — en la Tarea 8.4 ya no podréis comparar nada, porque baseline y augmented serán la misma lista |

> **Pregunta para pensar:** si en la Tarea 8.4 vuestra gráfica "Baseline vs Augmented" muestra dos líneas idénticas, ¿cuál de las variables de la tabla anterior sospecharíais primero que no se renombró?

### Una sola ejecución no es una prueba definitiva

En el Bloque 7.3 vimos que Loss y Dice pueden divergir (la "meseta engañosa"). Lo mismo aplica aquí: entrenar una vez cada configuración os da un dato, no una certeza estadística. Diferencias pequeñas en el Dice final entre baseline y augmented pueden deberse en parte a variabilidad normal del entrenamiento (inicialización aleatoria de pesos, orden de batches), no solo al augmentation en sí. Esto no invalida el experimento — solo hay que tenerlo en cuenta al interpretar una diferencia pequeña como "augmentation ganó" o "augmentation perdió".

### Resumen rápido

| Concepto | Idea principal |
|----------|-----------------|
| Reinicializar el modelo | Evita que la mejora se deba a "más épocas totales" en vez de al augmentation |
| Variable de control | Todo igual entre baseline y augmented excepto los datos de entrenamiento |
| Riesgo de copiar código | Olvidar renombrar variables sobrescribe silenciosamente los resultados del baseline |
| Una sola ejecución | Las diferencias pequeñas pueden ser variabilidad normal, no solo efecto del augmentation |

## Tarea 8.3: Re-entrenamiento con Augmentation
Ahora vamos a re-entrenar el modelo usando el nuevo DataLoader con augmentation.

### Instrucciones:
1. Crea un nuevo `BUSIDataset` (que permita transformaciones) usando `train_augment` en lugar del transform base
2. Crea un nuevo `DataLoader` para train (el de val puede quedarse igual)
3. Re-inicializa el modelo U-Net (para empezar desde cero)
4. Entrena durante `NUM_EPOCHS` épocas (usa el mismo código de entrenamiento de P7)
5. Guarda las métricas: `train_losses_aug`, `val_losses_aug`, `train_dices_aug`, `val_dices_aug`
6. Aplica el mismo patrón de checkpointing que en la P7, pero con `model_aug`: guarda los pesos localmente cada vez que mejore el Val Dice, y al terminar el bucle guarda también el checkpoint final (pesos + historial) en Google Drive — mismo `CHECKPOINT_DIR` de la P7, pero con un fichero distinto (por ejemplo `augmented_checkpoint.pth`) para no sobrescribir el del baseline. Comprueba primero si ese checkpoint ya existe en Drive: si es así, cárgalo y no reentrenes.
6. Aplica el mismo checkpointing que en la P7 (guardar el mejor modelo según Val Dice y recargarlo al final), pero usando `model_aug` y un fichero distinto (por ejemplo `best_model_augmented.pth`) para no sobrescribir el checkpoint del baseline.

> **Pista**: Puedes copiar el código de entrenamiento de P7 y modificar solo el DataLoader

In [ ]:
# ============================================================
# TAREA 8.3: RE-ENTRENAMIENTO CON AUGMENTATION
# ============================================================

from torch.utils.data import DataLoader

DRIVE_CHECKPOINT_PATH_AUG = CHECKPOINT_DIR / 'augmented_checkpoint.pth'

# Actualiza la clase BUSIDataset con soporte para Albumentations
# ESCRIBE TU CÓDIGO AQUÍ



# ESCRIBE TU CÓDIGO AQUÍ

# 1. Crear nuevo Dataset con augmentation
# train_dataset_aug = BUSIDataset(train_df, transform=train_augment)
# train_loader_aug = DataLoader(...)

# 2. Re-inicializar modelo
# model_aug = smp.Unet(...)
# model_aug = model_aug.to(DEVICE)

# 3. Optimizador
# optimizer_aug = optim.Adam(model_aug.parameters(), lr=1e-4)

# 4. Entrenar (copia el bucle de P7 pero usando train_loader_aug)
# Guarda: train_losses_aug, val_losses_aug, train_dices_aug, val_dices_aug

# 5. Checkpointing (igual que en la P7, pero con model_aug y DRIVE_CHECKPOINT_PATH_AUG):
#    - Si DRIVE_CHECKPOINT_PATH_AUG ya existe: cargarlo (pesos + historial) y no reentrenar
#    - Si no existe: best_val_dice_aug = -1 ; BEST_MODEL_PATH_AUG = 'best_model_augmented.pth'
#      dentro del bucle, si epoch_val_dice mejora, actualizar y guardar model_aug.state_dict()
#      al terminar: cargar el mejor local y guardar el checkpoint completo en Drive

print("Entrenando con augmentation...")

# Tu código aquí...

print("Entrenamiento con augmentation completado")

## Bloque 8.4: Leer una Comparativa de Curvas de Entrenamiento
### Cuatro paneles, cuatro preguntas distintas

La figura 2x2 no es simplemente "cuatro gráficas bonitas" — cada panel responde a una pregunta diferente, y hay que leerlos en ese orden:

| Panel | Pregunta que responde |
|-------|--------------------------|
| Train Loss | ¿Sigue aprendiendo el modelo sobre los datos que ve? |
| Train Dice | ¿Qué tan bien ajusta cada modelo los datos de entrenamiento? |
| Val Loss | ¿Cuál generaliza mejor según la función de pérdida? |
| Val Dice | ¿Cuál generaliza mejor según la métrica que realmente importa? |

### La métrica que de verdad decide: Val Dice

De los cuatro paneles, **Val Dice** es el que más se parece a lo que Kaggle va a puntuar sobre el test set (que no habéis visto). Train Loss y Train Dice solo os dicen qué tan bien memoriza cada modelo los datos que ya conoce — un valor alto ahí no garantiza nada sobre imágenes nuevas.

> **Idea clave:** si tuvierais que quedaros con un solo número para decidir qué modelo subir a Kaggle, sería el mejor Val Dice, no el mejor Train Dice.

### No miréis solo el valor final: mirad la brecha train-val

Recordad el Bloque 8.1: el síntoma de sobreajuste es que train y val se separan. Por eso la comparación más informativa no es "¿qué Val Dice final tiene cada modelo?", sino "¿qué distancia hay entre Train Dice y Val Dice en cada uno?":

| Modelo | Train Dice alto y Val Dice bajo (brecha grande) | Train Dice y Val Dice parecidos (brecha pequeña) |
|--------|----------------------------------------------------|-------------------------------------------------------|
| Interpretación | Sobreajuste: memoriza train, no generaliza | Generaliza mejor, aunque su Dice absoluto sea algo menor |

Es perfectamente posible que el modelo Augmented tenga un Train Dice **más bajo** que el Baseline (porque entrena con datos más difíciles y variados) y aun así sea el modelo que mejor generaliza, si su brecha train-val es menor.

### Ejes comparables: por qué fijar los mismos límites

Las instrucciones piden `axes[...].set_ylim(0, 1)` en los paneles de Dice. Esto no es estético: si cada subplot ajustara automáticamente su propio rango de eje Y, dos curvas con diferencias reales podrían parecer visualmente idénticas (o al revés, diferencias mínimas podrían parecer enormes) solo por el zoom automático de matplotlib. Para comparar de forma honesta, ambas curvas de un mismo panel deben compartir escala.

### Una diferencia pequeña no siempre es una victoria

Como vimos en el Bloque 8.3, una sola ejecución tiene variabilidad inherente. Si el Val Dice final del Augmented es, por ejemplo, 0.76 frente a 0.74 del Baseline, esa diferencia de 2 puntos puede deberse tanto al augmentation como al azar de esa ejecución concreta. Una diferencia consistente y visible durante varias épocas (no solo en el último punto) es una señal más fiable que comparar un único valor final.

> **Pregunta para pensar:** si al terminar veis que el Augmented tiene peor Val Dice que el Baseline con el mismo número de épocas, ¿descartaríais inmediatamente el augmentation, o hay algo en el Bloque 8.1 (sobre la dificultad añadida de entrenar con más variabilidad) que sugiera revisar antes de concluir nada?

### Resumen rápido

| Concepto | Idea principal |
|----------|-----------------|
| Cuatro paneles | Cada uno responde una pregunta distinta; no son intercambiables |
| Val Dice | La métrica más parecida a lo que evalúa Kaggle |
| Brecha train-val | Más informativa que el valor final aislado; brecha pequeña = mejor generalización |
| Ejes compartidos | `set_ylim` igual en ambas curvas evita comparaciones visualmente engañosas |
| Diferencia pequeña | Puede ser variabilidad de una sola ejecución, no una conclusión sólida |

## Tarea 8.4: Comparativa Baseline vs Augmented
Es hora de comparar los resultados. Vamos a visualizar las curvas de entrenamiento de ambos modelos lado a lado.

### Instrucciones:
1. Crea una figura de 2x2 con:
   - Gráfico 1: Train Loss (Baseline vs Augmented)
   - Gráfico 2: Val Loss (Baseline vs Augmented)
   - Gráfico 3: Train Dice (Baseline vs Augmented)
   - Gráfico 4: Val Dice (Baseline vs Augmented)
2. Usa diferentes colores para cada modelo
3. Añade leyendas, títulos y grid

> **CHECKPOINT P8.2:** Muestra al profesor la figura comparativa y explica las diferencias observadas

In [ ]:
# ============================================================
# TAREA 8.4: COMPARATIVA BASELINE VS AUGMENTED
# ============================================================
import matplotlib.pyplot as plt

# ESCRIBE TU CÓDIGO AQUÍ
# 1. Crea una figura de 2x2
# 2. Gráfico 1: Train Loss (train_losses vs train_losses_aug)
# 3. Gráfico 2: Val Loss (val_losses vs val_losses_aug)
# 4. Gráfico 3: Train Dice (train_dices vs train_dices_aug)
# 5. Gráfico 4: Val Dice (val_dices vs val_dices_aug)

# Tu código aquí...

plt.tight_layout()
plt.show()

# Imprimir resumen
print("\n" + "="*60)
print("RESUMEN COMPARATIVO")
print("="*60)
print(f"Baseline - Mejor Val Dice: {max(val_dices):.4f}")
print(f"Augmented - Mejor Val Dice: {max(val_dices_aug):.4f}")
print(f"Mejora: {max(val_dices_aug) - max(val_dices):+.4f}")